Notebook Objective : 
======================
Modular batch sequencing bandit + simulator for causal-informed delivery ordering.

Architecture
------------
- RewardOracle  (ABC)  ← pre-trained LightGBM per city, FROZEN, simulates the environment
- PolicyModel   (ABC)  ← online learner inside the bandit, updated from rewards
- BanditBase    (ABC)  ← selection policy wrapping a PolicyModel
- BatchEnvironment     ← state management + dynamic feature recomputation each step
- Simulator            ← runs one batch as one episode, collects metrics

Key design invariants
---------------------
- RewardOracle uses ALL features (static + dynamic). Goal: predictive accuracy.
- PolicyModel uses CAUSAL features only (φ_shared + φ_c per city). Goal: transferable policy.
- RewardOracle is never retrained during simulation. It is a frozen environment surrogate.
- PolicyModel is updated online after every delivery selection.
- Reward = −Δt  (bandit maximises, here we minimise duration).
- All distances are Euclidean (LaDe coordinates are affine-transformed planar, not real lat/lng).
- Dynamic features are recomputed from simulator state at every step, never read from the log.

Swapping components
-------------------
  Replace reward oracle:  subclass OracleBase, pass to BatchEnvironment.
  Replace policy model:   subclass PolicyModelBase, pass to SquareCBBandit / ThompsonSamplingBandit.
  Replace bandit:         subclass BanditBase, pass to Simulator.run_episode().

In [ ]:


from __future__ import annotations

import math
import warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.linear_model import SGDRegressor, Ridge
from sklearn.preprocessing import StandardScaler




# SECTION 1 — Causal feature config


In [ ]:
# Values are taken from actual CD-NOD / per-city PC output.
# Use the post-cyclic-encoding names from FINAL_FEATURES
# (i.e., hour_sin_dynamic / hour_cos_dynamic, not raw current_hour).
#
# φ_shared  → CD-NOD invariant parents → feeds SquareCB shared regression oracle
# φ_c       → per-city PC parents → augments φ_shared per city

CDNOD_FEATURES: List[str] = [
    # --- arm-level (per-delivery, vary across arms at each step) ---
    "dist_from_current",            # dynamic, recomputed each step — key discriminator
    "pickup_destination_distance",  # static
    "spatial_congestion_norm",      # static
    # --- context-level (shared across all arms at each step) ---
    "WSI",                          # static per batch
    "hour_sin_dynamic",             # dynamic, updates as route clock advances
    "hour_cos_dynamic",             # dynamic
    "last_duration",                # dynamic, = oracle output from previous step
    "remaining_orders",             # dynamic
    "batch_progress",               # dynamic
    "elapsed_route_time",           # dynamic
    "cumulative_distance",          # dynamic
    # TODO: add remaining confirmed CD-NOD invariant parents
]

PC_EXTRA_FEATURES_BY_CITY: Dict[str, List[str]] = {
    # Features confirmed by per-city PC but NOT already in CDNOD_FEATURES.
    # Leave empty if city-specific PC found no additional parents beyond consensus.
    "Shanghai": [
        # e.g. "typecode_cb",
        # TODO: add Shanghai-specific PC parents
    ],
    "Hangzhou": [
        # TODO: add Hangzhou-specific PC parents
    ],
    "Chongqing": [
        # e.g. "spatial_congestion_daily",  (stronger WSI edge confirmed)
        # TODO: add Chongqing-specific PC parents
    ],
}

def get_causal_features(city: str) -> List[str]:
    """φ_shared ∪ φ_c: full causal feature set for this city's policy oracle."""
    return CDNOD_FEATURES + [
        f for f in PC_EXTRA_FEATURES_BY_CITY.get(city, [])
        if f not in CDNOD_FEATURES
    ]


# SECTION 2 — Data structures


In [ ]:
@dataclass
class BatchState:
    """
    Mutable state of the simulator at the START of a delivery selection step.
    Represents everything the bandit can observe before choosing the next delivery.
    All values reflect the situation BEFORE step `step` is executed.
    """
    current_lat: float          # courier position after last completed delivery
    current_lng: float
    elapsed_route_time: float   # cumulative minutes elapsed in this batch
    cumulative_distance: float  # cumulative Euclidean distance travelled
    last_duration: float        # Δt of the most recently completed delivery (0 if first)
    step: int                   # 0-indexed step counter (= number of completed deliveries)
    batch_size: int             # total deliveries in this batch
    start_hour: float           # fractional hour at batch start (for current_hour computation)
    batch_id: str               # for logging

    @property
    def remaining_orders(self) -> int:
        return self.batch_size - self.step

    @property
    def batch_progress(self) -> float:
        return self.step / max(self.batch_size, 1)

    @property
    def current_hour(self) -> float:
        return self.start_hour + self.elapsed_route_time / 60.0

    @property
    def hour_sin_dynamic(self) -> float:
        return math.sin(self.current_hour * 2 * math.pi / 24)

    @property
    def hour_cos_dynamic(self) -> float:
        return math.cos(self.current_hour * 2 * math.pi / 24)


@dataclass
class DeliveryRecord:
    """
    One delivery within a batch.  static_features holds all pre-computed
    static values (structural + operational + environmental + temporal) exactly
    as they appear in FINAL_FEATURES from the oracle training script.
    poi_lat / poi_lng are stored separately because the environment uses them
    to compute dist_from_current at each step.
    """
    delivery_id: str
    poi_lat: float
    poi_lng: float
    static_features: Dict[str, float]   # key = feature name, value = scalar
    logged_rank: int = -1               # batch_rank_actual from log, for LoggedPolicy


@dataclass
class EpisodeResult:
    """Metrics collected from one batch episode under one policy."""
    city: str
    batch_id: str
    policy_name: str
    sequence: List[str]                  # ordered delivery_ids
    predicted_durations: List[float]     # oracle Δt per step
    total_duration: float                # Σ predicted_durations
    mean_duration: float
    step_rewards: List[float]            # −Δt per step (what bandit maximised)


# SECTION 3 — Reward Oracle (environment surrogate — NOT the policy)


**Predictive step**: At each step of a delivery sequence, the simulator asks: "If the courier delivers order X next, how long will it take?" The LightGBM model predicts this incremental duration 
(Δt).
**Reward**: The bandit policy receives a reward of −Δt
(since the goal is to minimize total delivery time, maximizing reward is equivalent to minimizing duration).

In [ ]:
class OracleBase(ABC):
    """
    Pre-trained model that predicts incremental duration Δt for a delivery
    given its full feature vector (static + dynamic, all features).
    This is the ENVIRONMENT surrogate.  Never updated during simulation.
    """
    @abstractmethod
    def predict(self, feature_vector: np.ndarray) -> float:
        """Single prediction.  feature_vector must match training feature order."""

    @abstractmethod
    def predict_batch(self, feature_matrix: np.ndarray) -> np.ndarray:
        """Vectorised prediction over multiple arms at once."""

    @abstractmethod
    def feature_names(self) -> List[str]:
        """Returns ordered feature names that this oracle expects."""


class LightGBMOracle(OracleBase):
    """
    Wraps a pre-trained LightGBM model saved via joblib.
    Drop-in replacement: swap model_path or swap entire class with XGBoostOracle, etc.
    """
    def __init__(self, model_path: str, feature_names_list: List[str]):
        self._model = joblib.load(model_path)
        self._feature_names = feature_names_list

    def predict(self, feature_vector: np.ndarray) -> float:
        return float(self._model.predict(feature_vector.reshape(1, -1))[0])

    def predict_batch(self, feature_matrix: np.ndarray) -> np.ndarray:
        return self._model.predict(feature_matrix)

    def feature_names(self) -> List[str]:
        return self._feature_names


# SECTION 4 — Policy Model (online learner inside the bandit)


In [ ]:
class PolicyModelBase(ABC):
    """
    Online regression model used by the bandit to SCORE arms.
    Receives CAUSAL features only (φ_shared + φ_c).
    Gets updated after each delivery selection with the observed reward.
    This is separate from the reward oracle — it learns from experience.
    """
    @abstractmethod
    def predict(self, features: np.ndarray) -> float:
        """Score one arm feature vector."""

    @abstractmethod
    def predict_batch(self, features: np.ndarray) -> np.ndarray:
        """Score all remaining arms at once.  features: (n_arms, n_features)."""

    @abstractmethod
    def update(self, features: np.ndarray, reward: float) -> None:
        """Online update with observed (features, reward) pair."""


class OnlineSGDPolicyModel(PolicyModelBase):
    """
    Online linear regression via SGD.  Fast, streaming, no batch refit needed.
    Scales well to a simulation that processes thousands of delivery steps.
    Swap for RidgePolicyModel or NeuralPolicyModel without changing bandit code.
    """
    def __init__(self, n_features: int, learning_rate: float = 0.01,
                 alpha: float = 1e-4, random_state: int = 42):
        self._scaler = StandardScaler()
        self._scaler_fitted = False
        self._model = SGDRegressor(
            loss="squared_error",
            learning_rate="constant",
            eta0=learning_rate,
            alpha=alpha,
            random_state=random_state,
            warm_start=True,
        )
        self._n_features = n_features
        # cold-start buffer: collect a few samples before first fit
        self._buffer_X: List[np.ndarray] = []
        self._buffer_y: List[float] = []
        self._min_samples_before_fit = max(10, n_features)

    def _ensure_fitted(self) -> bool:
        if len(self._buffer_y) < self._min_samples_before_fit:
            return False
        if not self._scaler_fitted:
            X = np.vstack(self._buffer_X)
            self._scaler.fit(X)
            self._scaler_fitted = True
            Xs = self._scaler.transform(X)
            self._model.partial_fit(Xs, np.array(self._buffer_y))
        return True

    def predict(self, features: np.ndarray) -> float:
        if not self._ensure_fitted():
            return 0.0
        return float(self._model.predict(self._scaler.transform(features.reshape(1, -1)))[0])

    def predict_batch(self, features: np.ndarray) -> np.ndarray:
        if not self._ensure_fitted():
            return np.zeros(len(features))
        return self._model.predict(self._scaler.transform(features))

    def update(self, features: np.ndarray, reward: float) -> None:
        if not self._scaler_fitted:
            self._buffer_X.append(features)
            self._buffer_y.append(reward)
            self._ensure_fitted()
            return
        Xs = self._scaler.transform(features.reshape(1, -1))
        self._model.partial_fit(Xs, np.array([reward]))


class RidgePolicyModel(PolicyModelBase):
    """
    Incremental Ridge regression using the Sherman-Morrison rank-1 update.
    Closed-form, no learning rate to tune.  Good alternative to SGD.
    """
    def __init__(self, n_features: int, alpha: float = 1.0):
        self._alpha = alpha
        n = n_features
        self._A = alpha * np.eye(n)      # A = αI + Σ z_t z_t^T
        self._b = np.zeros(n)            # b = Σ r_t z_t
        self._theta = np.zeros(n)        # θ = A^{-1} b
        self._A_inv = (1.0 / alpha) * np.eye(n)

    def _update_inv(self, z: np.ndarray) -> None:
        # Sherman-Morrison: (A + z z^T)^{-1} = A^{-1} - (A^{-1} z z^T A^{-1}) / (1 + z^T A^{-1} z)
        Az = self._A_inv @ z
        denom = 1.0 + z @ Az
        self._A_inv -= np.outer(Az, Az) / denom

    def predict(self, features: np.ndarray) -> float:
        return float(features @ self._theta)

    def predict_batch(self, features: np.ndarray) -> np.ndarray:
        return features @ self._theta

    def update(self, features: np.ndarray, reward: float) -> None:
        z = features
        self._update_inv(z)
        self._b += reward * z
        self._theta = self._A_inv @ self._b


# SECTION 5 — Bandit policies


In [ ]:
class BanditBase(ABC):
    """Abstract bandit.  arm_features: (n_arms, n_causal_features)."""
    @abstractmethod
    def select_arm(self, arm_features: np.ndarray) -> int:
        """Return index into arm_features of the chosen arm."""

    @abstractmethod
    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        """Update policy with observed reward."""

    @property
    def name(self) -> str:
        return self.__class__.__name__


class SquareCBBandit(BanditBase):
    """
    SquareCB (Foster & Rakhlin, 2020) — contextual bandit reduction to regression.
    Works with any PolicyModel.  No linearity or sub-Gaussian noise assumption.

    Selection rule:
        r̂[a] = policy_model.predict(features[a])
        r_max = max(r̂)
        w[a]  = 1 / (γ + (r_max − r̂[a])²)
        p[a]  = w[a] / Σ w        ← softmax over inverse squared gaps

    Exploration: smaller γ → more concentrated on best arm (greedy).
                 larger γ → more uniform exploration.
    Sweep γ ∈ {0.1, 0.5, 1.0, 5.0, 10.0} per city.
    """
    def __init__(self, policy_model: PolicyModelBase, gamma: float = 1.0,
                 random_state: int = 42):
        self._model = policy_model
        self._gamma = gamma
        self._rng = np.random.default_rng(random_state)
        self._n_selections = 0

    @property
    def name(self) -> str:
        return f"SquareCB(γ={self._gamma})"

    def select_arm(self, arm_features: np.ndarray) -> int:
        r_hat = self._model.predict_batch(arm_features)          # (n_arms,)
        r_max = r_hat.max()
        weights = 1.0 / (self._gamma + (r_max - r_hat) ** 2)
        probs = weights / weights.sum()
        return int(self._rng.choice(len(r_hat), p=probs))

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        self._model.update(chosen_arm_features, reward)
        self._n_selections += 1


class ThompsonSamplingBandit(BanditBase):
    """
    Linear Thompson Sampling (Agrawal & Goyal, 2013).
    Maintains a Gaussian posterior over θ and samples at each step.
    Assumes linear reward — included as a comparison model, acknowledging this
    is an approximation vs SquareCB's nonparametric approach.

    Uses the same RidgePolicyModel internals for posterior update.
    Separate from SquareCB so the same policy can be compared side-by-side.
    """
    def __init__(self, n_features: int, v_sq: float = 0.25,
                 sigma_sq: float = 1.0, random_state: int = 42):
        self._v_sq = v_sq
        self._sigma_sq = sigma_sq
        self._rng = np.random.default_rng(random_state)
        self._ridge = RidgePolicyModel(n_features, alpha=1.0 / sigma_sq)

    @property
    def name(self) -> str:
        return f"ThompsonSampling(v²={self._v_sq})"

    def select_arm(self, arm_features: np.ndarray) -> int:
        # Sample θ̃ ~ N(μ, v² Σ)
        theta_sample = self._rng.multivariate_normal(
            mean=self._ridge._theta,
            cov=self._v_sq * self._ridge._A_inv,
        )
        scores = arm_features @ theta_sample
        return int(np.argmax(scores))

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        self._ridge.update(chosen_arm_features, reward)


# SECTION 6 — Baseline policies (no learning)


In [ ]:
class RandomBandit(BanditBase):
    """Uniform random arm selection.  Floor baseline."""
    def __init__(self, random_state: int = 42):
        self._rng = np.random.default_rng(random_state)

    @property
    def name(self) -> str:
        return "Random"

    def select_arm(self, arm_features: np.ndarray) -> int:
        return int(self._rng.integers(0, len(arm_features)))

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        pass


class GreedyNNBandit(BanditBase):
    """
    Greedy nearest-neighbour: always pick the delivery closest to current position.
    dist_from_current must be the FIRST feature in arm_features (index 0).
    Ensure CDNOD_FEATURES[0] == 'dist_from_current' or adjust dist_idx below.
    """
    def __init__(self, dist_feature_idx: int = 0):
        self._dist_idx = dist_feature_idx

    @property
    def name(self) -> str:
        return "GreedyNN"

    def select_arm(self, arm_features: np.ndarray) -> int:
        # arm_features[:, dist_idx] = dist_from_current for each arm
        # Maximise reward = −duration ≈ minimise distance → argmin distance
        # But reward is negative duration, so we want argmin(dist_from_current)
        return int(np.argmin(arm_features[:, self._dist_idx]))

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        pass


class GreedyOracleBandit(BanditBase):
    """
    At each step, calls the reward oracle for ALL remaining arms and picks the
    one with the lowest predicted duration.  Approximate upper-bound ceiling
    (greedy oracle, not globally optimal).  Used to compute NPG.
    Requires access to the environment — injected at construction.
    """
    def __init__(self):
        # _scores is set by BatchEnvironment.compute_all_oracle_scores()
        # before select_arm is called each step.
        self._scores: Optional[np.ndarray] = None

    @property
    def name(self) -> str:
        return "GreedyOracle"

    def set_oracle_scores(self, scores: np.ndarray) -> None:
        """Called by the simulator just before select_arm at each step."""
        self._scores = scores

    def select_arm(self, arm_features: np.ndarray) -> int:
        if self._scores is None:
            return 0
        return int(np.argmax(self._scores))  # scores = −duration, argmax = min duration

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        pass


class LoggedOrderBandit(BanditBase):
    """
    Replays the actual logged courier ordering (batch_rank_actual).
    Used to score the human-expert baseline through the same oracle.
    """
    @property
    def name(self) -> str:
        return "LoggedOrder"

    def select_arm(self, arm_features: np.ndarray) -> int:
        # Selection is overridden by Simulator using delivery.logged_rank.
        # This method is only a fallback.
        return 0

    def update(self, chosen_arm_features: np.ndarray, reward: float) -> None:
        pass


# SECTION 7 — BatchEnvironment


In a real delivery run, the features of remaining packages change as the courier moves. The BatchEnvironment class manages this simulator state:

When a delivery is completed, the courier's location is updated to that customer's coordinates.
The elapsed time, current hour, and cumulative distance are advanced.
For the next step, dynamic features (e.g., dist_from_current, remaining_orders, batch_progress, elapsed_route_time, cumulative_distance) are recomputed from the simulator state for all remaining candidate deliveries.

In [ ]:
class BatchEnvironment:
    """
    Manages simulator state and feature computation for one batch episode.

    Responsibilities:
      - Initialise state at batch start.
      - For each step: compute dynamic features for ALL remaining deliveries.
      - Assemble two feature vectors per delivery:
          oracle_vec  → ALL features in oracle training order (for reward oracle)
          policy_vec  → CAUSAL features only (for bandit policy model)
      - Execute a delivery selection, update state, return reward.

    Dynamic feature computation is fully self-contained here: no logged values
    are carried over.  This ensures train/inference consistency.
    """

    def __init__(
        self,
        oracle: OracleBase,
        oracle_feature_names: List[str],    # FINAL_FEATURES from training script
        causal_feature_names: List[str],    # get_causal_features(city)
    ):
        self._oracle = oracle
        self._oracle_feats = oracle_feature_names
        self._causal_feats = causal_feature_names

    # ------------------------------------------------------------------
    # Episode initialisation
    # ------------------------------------------------------------------

    def reset(
        self,
        deliveries: List[DeliveryRecord],
        start_lat: float,
        start_lng: float,
        start_hour: float,
        batch_id: str,
    ) -> Tuple[BatchState, List[DeliveryRecord]]:
        """
        Returns initial state and a mutable copy of the delivery list.
        start_lat/lng = courier position at batch dispatch (use mean receipt coords
        or dedicated pickup coords if available).
        """
        state = BatchState(
            current_lat=start_lat,
            current_lng=start_lng,
            elapsed_route_time=0.0,
            cumulative_distance=0.0,
            last_duration=0.0,
            step=0,
            batch_size=len(deliveries),
            start_hour=start_hour,
            batch_id=batch_id,
        )
        return state, list(deliveries)

    # ------------------------------------------------------------------
    # Dynamic feature computation
    # ------------------------------------------------------------------

    @staticmethod
    def euclidean(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
        """
        Planar Euclidean distance in coordinate units.
        Correct for LaDe's affine-transformed space (NOT haversine).
        """
        return math.sqrt((lat2 - lat1) ** 2 + (lng2 - lng1) ** 2)

    def _dynamic_features(
        self,
        delivery: DeliveryRecord,
        state: BatchState,
    ) -> Dict[str, float]:
        """
        Compute all dynamic (position-dependent) features for one candidate delivery
        given the current simulator state.  These are the features that must NOT be
        read from the log — they depend on which ordering the bandit has proposed.
        """
        dist = self.euclidean(
            state.current_lat, state.current_lng,
            delivery.poi_lat, delivery.poi_lng,
        )
        return {
            "dist_from_current":    dist,
            "remaining_orders":     float(state.remaining_orders),
            "batch_progress":       state.batch_progress,
            "elapsed_route_time":   state.elapsed_route_time,
            "last_duration":        state.last_duration,
            "current_hour":         state.current_hour,
            "cumulative_distance":  state.cumulative_distance,
            # cyclic encoding of the updated hour (matches training feature names)
            "hour_sin_dynamic":     state.hour_sin_dynamic,
            "hour_cos_dynamic":     state.hour_cos_dynamic,
        }

    def _assemble_oracle_vector(
        self,
        delivery: DeliveryRecord,
        dynamic: Dict[str, float],
    ) -> np.ndarray:
        """
        Assemble feature vector for the reward oracle in exactly the order
        oracle_feature_names specifies.  Missing static features raise KeyError
        early, not silently during prediction.
        """
        merged = {**delivery.static_features, **dynamic}
        return np.array([merged[f] for f in self._oracle_feats], dtype=np.float32)

    def _assemble_policy_vector(
        self,
        delivery: DeliveryRecord,
        dynamic: Dict[str, float],
    ) -> np.ndarray:
        """
        Assemble feature vector for the bandit policy model.
        Uses CAUSAL features only — causal_feature_names defines the order.
        """
        merged = {**delivery.static_features, **dynamic}
        return np.array([merged[f] for f in self._causal_feats], dtype=np.float32)

    # ------------------------------------------------------------------
    # Per-step interface
    # ------------------------------------------------------------------

    def get_arm_vectors(
        self,
        remaining: List[DeliveryRecord],
        state: BatchState,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        For all remaining deliveries, compute:
          oracle_matrix : (n_arms, n_oracle_features)   — for reward oracle
          policy_matrix : (n_arms, n_causal_features)   — for bandit policy
        """
        oracle_rows, policy_rows = [], []
        for d in remaining:
            dyn = self._dynamic_features(d, state)
            oracle_rows.append(self._assemble_oracle_vector(d, dyn))
            policy_rows.append(self._assemble_policy_vector(d, dyn))
        return np.vstack(oracle_rows), np.vstack(policy_rows)

    def score_all_arms_oracle(
        self,
        oracle_matrix: np.ndarray,
    ) -> np.ndarray:
        """
        Call reward oracle for all arms at once.
        Returns scores = −predicted_duration (so argmax → min duration).
        Used by GreedyOracleBandit and for metric computation.
        """
        durations = self._oracle.predict_batch(oracle_matrix)
        return -durations   # reward = −Δt

    def step(
        self,
        state: BatchState,
        chosen: DeliveryRecord,
        chosen_oracle_vec: np.ndarray,
    ) -> Tuple[BatchState, float]:
        """
        Execute selecting `chosen` as the next delivery.
        Returns updated state and the reward (= −predicted_duration).
        """
        predicted_duration = self._oracle.predict(chosen_oracle_vec)
        reward = -predicted_duration

        dist_hop = self.euclidean(
            state.current_lat, state.current_lng,
            chosen.poi_lat, chosen.poi_lng,
        )

        new_state = BatchState(
            current_lat=chosen.poi_lat,
            current_lng=chosen.poi_lng,
            elapsed_route_time=state.elapsed_route_time + predicted_duration,
            cumulative_distance=state.cumulative_distance + dist_hop,
            last_duration=predicted_duration,
            step=state.step + 1,
            batch_size=state.batch_size,
            start_hour=state.start_hour,
            batch_id=state.batch_id,
        )
        return new_state, reward


# SECTION 8 — Simulator


In [ ]:
class Simulator:
    """
    Runs one batch as one episode.
    For every remaining delivery at each step:
      1. Compute oracle + policy feature vectors.
      2. (If GreedyOracleBandit) inject oracle scores.
      3. Bandit selects an arm.
      4. Environment executes the selection → (new_state, reward).
      5. Bandit updates its policy model.
    """

    def __init__(self, env: BatchEnvironment):
        self._env = env

    def run_episode(
        self,
        deliveries: List[DeliveryRecord],
        bandit: BanditBase,
        city: str,
        start_lat: float,
        start_lng: float,
        start_hour: float,
        batch_id: str = "",
    ) -> EpisodeResult:
        state, remaining = self._env.reset(
            deliveries, start_lat, start_lng, start_hour, batch_id
        )

        sequence: List[str] = []
        predicted_durations: List[float] = []
        step_rewards: List[float] = []

        while remaining:
            oracle_mat, policy_mat = self._env.get_arm_vectors(remaining, state)

            # Special handling for LoggedOrderBandit: follow actual rank
            if isinstance(bandit, LoggedOrderBandit):
                arm_idx = self._logged_order_index(remaining)
            elif isinstance(bandit, GreedyOracleBandit):
                oracle_scores = self._env.score_all_arms_oracle(oracle_mat)
                bandit.set_oracle_scores(oracle_scores)
                arm_idx = bandit.select_arm(policy_mat)
            else:
                arm_idx = bandit.select_arm(policy_mat)

            chosen = remaining[arm_idx]
            state, reward = self._env.step(state, chosen, oracle_mat[arm_idx])

            bandit.update(policy_mat[arm_idx], reward)

            sequence.append(chosen.delivery_id)
            predicted_durations.append(-reward)   # convert back to positive duration
            step_rewards.append(reward)
            remaining.pop(arm_idx)

        total = sum(predicted_durations)
        return EpisodeResult(
            city=city,
            batch_id=batch_id,
            policy_name=bandit.name,
            sequence=sequence,
            predicted_durations=predicted_durations,
            total_duration=total,
            mean_duration=total / max(len(predicted_durations), 1),
            step_rewards=step_rewards,
        )

    @staticmethod
    def _logged_order_index(remaining: List[DeliveryRecord]) -> int:
        """Return index of the delivery with the lowest logged_rank."""
        return int(min(range(len(remaining)), key=lambda i: remaining[i].logged_rank))


# SECTION 9 — Metrics


In [ ]:
def compute_episode_metrics(
    bandit_results: List[EpisodeResult],
    baseline_results: Dict[str, List[EpisodeResult]],
) -> pd.DataFrame:
    """
    Given a list of bandit episode results and a dict of baseline results
    (keyed by policy name), compute per-city aggregate metrics.

    Returns a DataFrame with columns:
      city, policy, mean_total_duration, delta_pct_vs_ranketpa,
      delta_pct_vs_random, NPG, n_batches
    """
    all_results = list(bandit_results)
    for results in baseline_results.values():
        all_results.extend(results)

    df = pd.DataFrame([
        {
            "city": r.city,
            "policy": r.policy_name,
            "batch_id": r.batch_id,
            "total_duration": r.total_duration,
        }
        for r in all_results
    ])

    agg = df.groupby(["city", "policy"])["total_duration"].mean().reset_index()
    agg.columns = ["city", "policy", "mean_total_duration"]

    rows = []
    for city in agg["city"].unique():
        city_agg = agg[agg["city"] == city].set_index("policy")["mean_total_duration"]
        random_eta = city_agg.get("Random", np.nan)
        oracle_eta = city_agg.get("GreedyOracle", np.nan)
        ranketpa_eta = city_agg.get("GreedyOracle", np.nan)  # use GreedyOracle as RANKETPA proxy

        for policy, mean_eta in city_agg.items():
            delta_ranketpa = (
                (ranketpa_eta - mean_eta) / ranketpa_eta * 100
                if not np.isnan(ranketpa_eta) and ranketpa_eta != 0 else np.nan
            )
            delta_random = (
                (random_eta - mean_eta) / random_eta * 100
                if not np.isnan(random_eta) and random_eta != 0 else np.nan
            )
            npg = (
                (mean_eta - oracle_eta) / (random_eta - oracle_eta)
                if not np.isnan(oracle_eta) and not np.isnan(random_eta)
                   and random_eta != oracle_eta else np.nan
            )
            rows.append({
                "city": city, "policy": policy,
                "mean_total_duration": round(mean_eta, 3),
                "delta_pct_vs_greedy_oracle": round(delta_ranketpa, 2),
                "delta_pct_vs_random": round(delta_random, 2),
                "NPG": round(npg, 4) if not np.isnan(npg) else np.nan,
            })

    return pd.DataFrame(rows).sort_values(["city", "mean_total_duration"])


def compute_cumulative_regret(
    bandit_results: List[EpisodeResult],
    oracle_results: List[EpisodeResult],
) -> pd.DataFrame:
    """
    Computes cumulative regret = Σ (oracle_reward − bandit_reward) per batch.
    Both lists must be aligned by batch_id.
    """
    oracle_map = {r.batch_id: sum(r.step_rewards) for r in oracle_results}
    rows = []
    cum_regret = 0.0
    for i, r in enumerate(bandit_results):
        oracle_val = oracle_map.get(r.batch_id, 0.0)
        regret = oracle_val - sum(r.step_rewards)
        cum_regret += regret
        rows.append({
            "batch_index": i,
            "city": r.city,
            "batch_id": r.batch_id,
            "instant_regret": regret,
            "cumulative_regret": cum_regret,
        })
    return pd.DataFrame(rows)


# SECTION 10 — Data loading helpers


In [ ]:
def load_batches_from_parquet(
    path: str,
    oracle_feature_names: List[str],
    city: str,
    max_batches: Optional[int] = None,
) -> List[Tuple[str, List[DeliveryRecord], float, float, float]]:
    """
    Load evaluation batches from a processed parquet file (output of 07_reward_oracle.py).
    Returns list of (batch_id, [DeliveryRecord...], start_lat, start_lng, start_hour).

    Expects columns: batch_id, batch_rank_actual, poi_lat, poi_lng,
    receipt_time (or start_hour pre-computed), + all FINAL_FEATURES.
    """
    df = pl.read_parquet(path)

    # Static features = oracle features minus dynamic features
    dynamic_names = {
        "dist_from_current", "remaining_orders", "batch_progress",
        "elapsed_route_time", "last_duration", "current_hour",
        "cumulative_distance", "hour_sin_dynamic", "hour_cos_dynamic",
    }
    static_names = [f for f in oracle_feature_names if f not in dynamic_names]

    batches = []
    for batch_id, group in df.group_by("batch_id"):
        group_sorted = group.sort("batch_rank_actual")
        rows = group_sorted.to_dicts()

        deliveries = []
        for row in rows:
            static_feats = {k: float(row.get(k, 0.0) or 0.0) for k in static_names}
            deliveries.append(DeliveryRecord(
                delivery_id=str(row.get("order_id", row.get("delivery_id", ""))),
                poi_lat=float(row["poi_lat"]),
                poi_lng=float(row["poi_lng"]),
                static_features=static_feats,
                logged_rank=int(row["batch_rank_actual"]),
            ))

        # Batch start: use first row's receipt_time-derived hour
        first = rows[0]
        if "start_hour" in first:
            start_hour = float(first["start_hour"])
        elif "receipt_time" in first:
            ts = pd.to_datetime(first["receipt_time"])
            start_hour = ts.hour + ts.minute / 60.0
        else:
            start_hour = 9.0   # fallback

        # Courier start position: use first delivery's logged previous position
        # approximation: mean POI position as centroid proxy for batch start
        mean_lat = float(np.mean([d.poi_lat for d in deliveries]))
        mean_lng = float(np.mean([d.poi_lng for d in deliveries]))

        batches.append((str(batch_id), deliveries, mean_lat, mean_lng, start_hour))
        if max_batches and len(batches) >= max_batches:
            break

    return batches


# SECTION 11 — Experiment runner


In [ ]:
def run_city_experiment(
    city: str,
    oracle_model_path: str,
    data_path: str,
    oracle_feature_names: List[str],
    gamma_values: List[float] = (0.5, 1.0, 5.0),
    v_sq_values: List[float] = (0.1, 0.25, 1.0),
    max_eval_batches: Optional[int] = None,
    random_state: int = 42,
) -> Dict[str, object]:
    """
    Full experiment for one city:
      - Load oracle + batches
      - Run SquareCB(γ), ThompsonSampling(v²), and all baselines
      - Return metrics DataFrame + cumulative regret DataFrame

    Ablation A1 (causal vs full features in bandit) is achieved by calling
    this function twice: once with causal_features=get_causal_features(city),
    once with causal_features=oracle_feature_names (full set).
    """
    print(f"\n{'='*60}")
    print(f"  City: {city}")
    print(f"{'='*60}")

    causal_feats = get_causal_features(city)
    n_causal = len(causal_feats)
    print(f"  Causal features ({n_causal}): {causal_feats}")

    # Load oracle and batches
    oracle = LightGBMOracle(oracle_model_path, oracle_feature_names)
    env = BatchEnvironment(oracle, oracle_feature_names, causal_feats)
    sim = Simulator(env)

    batches = load_batches_from_parquet(data_path, oracle_feature_names, city, max_eval_batches)
    print(f"  Evaluation batches loaded: {len(batches)}")

    # Define all policies to evaluate
    policies = {
        "Random": RandomBandit(random_state),
        "GreedyNN": GreedyNNBandit(dist_feature_idx=causal_feats.index("dist_from_current")),
        "GreedyOracle": GreedyOracleBandit(),
        "LoggedOrder": LoggedOrderBandit(),
    }
    # SquareCB variants (γ sweep)
    for g in gamma_values:
        name = f"SquareCB(γ={g})"
        policies[name] = SquareCBBandit(
            OnlineSGDPolicyModel(n_causal), gamma=g, random_state=random_state
        )
    # Thompson Sampling variants (v² sweep)
    for v in v_sq_values:
        name = f"ThompsonSampling(v²={v})"
        policies[name] = ThompsonSamplingBandit(n_causal, v_sq=v, random_state=random_state)

    # Run all policies
    all_episode_results: Dict[str, List[EpisodeResult]] = {p: [] for p in policies}

    for batch_id, deliveries, slat, slng, shour in batches:
        for policy_name, bandit in policies.items():
            result = sim.run_episode(
                list(deliveries),   # fresh copy per policy
                bandit, city, slat, slng, shour, batch_id,
            )
            all_episode_results[policy_name].append(result)

    # Compute metrics
    bandit_names = [k for k in policies if k not in ("Random", "GreedyOracle", "LoggedOrder", "GreedyNN")]
    bandit_results_flat = [r for k in bandit_names for r in all_episode_results[k]]
    metrics_df = compute_episode_metrics(
        bandit_results_flat,
        {k: v for k, v in all_episode_results.items()},
    )

    regret_df = compute_cumulative_regret(
        all_episode_results.get(f"SquareCB(γ={gamma_values[1]})", []),
        all_episode_results.get("GreedyOracle", []),
    )

    print(f"\n  Results for {city}:")
    print(metrics_df[metrics_df["city"] == city].to_string(index=False))

    return {
        "metrics": metrics_df,
        "regret": regret_df,
        "episode_results": all_episode_results,
    }


# SECTION 12 — Main


In [ ]:
if __name__ == "__main__":
    # -----------------------------------------------------------------------
    # Config — adjust paths and feature lists
    # -----------------------------------------------------------------------
    from pathlib import Path

    # FINAL_FEATURES from 07_reward_oracle.py (post cyclic-encoding, no current_hour)
    # Import or redefine here.  Must match oracle training exactly.
    # Example — replace with your actual list:
    FINAL_FEATURES_EXAMPLE = (
        # structural
        ["pickup_destination_distance", "batch_size", "batch_rank_dispatch",
         "same_aoi_share_in_batch", "isolated_delivery", "distance_to_batch_centroid",
         "typecode_cb",
         # operational
         "courier_eta_ewm", "gps_points", "speed_mean_15m", "speed_std_15m",
         "distance_travelled_15m", "coverage_ratio", "gps_gap_min",
         "idle_fraction", "is_trajectory_available",
         # environment
         "WSI", "temperature_2m", "precipitation", "windspeed_10m",
         "spatial_congestion_daily", "spatial_congestion_norm",
         # temporal (raw static)
         "hour_sin", "hour_cos", "is_weekend", "is_holiday", "is_holiday_eve",
         # dynamic (recomputed each step)
         "dist_from_current", "remaining_orders", "batch_progress",
         "elapsed_route_time", "last_duration", "cumulative_distance",
         # cyclic encoding replaces current_hour
         "hour_sin_dynamic", "hour_cos_dynamic"]
    )

    ROOT = Path("data")
    WEIGHTS = ROOT / "LightGBMweights"

    city_configs = {
        "Shanghai": {
            "oracle_path": str(WEIGHTS / "oracle_sh.pkl"),
            "data_path": str(ROOT / "delivery_features_shanghai.parquet"),
        },
        "Hangzhou": {
            "oracle_path": str(WEIGHTS / "oracle_hz.pkl"),
            "data_path": str(ROOT / "delivery_features_hangzhou.parquet"),
        },
        "Chongqing": {
            "oracle_path": str(WEIGHTS / "oracle_cq.pkl"),
            "data_path": str(ROOT / "delivery_features_chongqing.parquet"),
        },
    }

    all_city_results = {}
    for city, cfg in city_configs.items():
        all_city_results[city] = run_city_experiment(
            city=city,
            oracle_model_path=cfg["oracle_path"],
            data_path=cfg["data_path"],
            oracle_feature_names=FINAL_FEATURES_EXAMPLE,
            gamma_values=[0.5, 1.0, 5.0],
            v_sq_values=[0.1, 0.25, 1.0],
            max_eval_batches=100,   # set to e.g. 100 for a quick test run
        )

    # Combine across cities
    combined_metrics = pd.concat(
        [r["metrics"] for r in all_city_results.values()], ignore_index=True
    )
    print("\n\n=== Combined results across all cities ===")
    print(combined_metrics.to_string(index=False))
